<a href="https://colab.research.google.com/github/shivamdubey715/ANN-Classification/blob/main/Housing_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()
df_test.info()

In [ ]:
df.drop('Id', axis=1, inplace=True)
df_test.drop('Id', axis=1, inplace=True)

In [ ]:
plt.figure(figsize=(15, 16))
plt.hist(df['SalePrice'], bins=30, color ='blue', edgecolor='black')
plt.title('Distribution of Sale Price')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import seaborn as sns
sns.boxplot(x=df['SalePrice'])

In [ ]:
numerical_features = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = numerical_features.corr()
corr_with_target = corr_matrix['SalePrice'].sort_values(ascending=False)
print(corr_with_target)

In [ ]:
#visualize the correlation
plt.figure(figsize=(10, 8))
corr_with_target.drop('SalePrice').plot(kind='bar', color='red')
plt.title('Correlation with Sale Price')
plt.xlabel('Features')
plt.ylabel('Correlation')
plt.show()

In [ ]:
mean_price = df['SalePrice'].mean()
median_price = df['SalePrice'].median()
mode_price = df['SalePrice'].mode()[0]
std_price = df['SalePrice'].std()
var_price = df['SalePrice'].var()
min_price = df['SalePrice'].min()
max_price = df['SalePrice'].max()
range_price = max_price - min_price
q1 = df['SalePrice'].quantile(0.25)
q3 = df['SalePrice'].quantile(0.75)
iqr = q3 - q1
skewness = df['SalePrice'].skew()
kurtosis = df['SalePrice'].kurt()

print('Mean:', mean_price)
print('Median:', median_price)
print('Mode:', mode_price)
print('Standard Deviation:', std_price)
print('Variance:', var_price)
print('Minimum:', min_price)
print('Maximum:', max_price)
print('Range:', range_price)
print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Skewness:', skewness)
print('Kurtosis:', kurtosis)



In [ ]:
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = df[(df['SalePrice']<lower_bound) | (df['SalePrice']>upper_bound)]
print(len(outliers))

In [ ]:
column_types_count = df.dtypes.value_counts()
print(column_types_count)

Creating aggregated feature like TotalSF and TotalBathrooms can enhance model performance by reducing dimentionality and make pattern easier to learn. also, these features often correlate strongly with house price.

In [ ]:
#check if there's missing values in test data
test_missing_values=df_test.isnull().sum()
test_missing_values=test_missing_values[test_missing_values>0]
print(test_missing_values)


In [ ]:
#Data Cleaning - handling missing values
fill_values = {}
for col in df.columns:
    if df[col].dtype == 'object':
        fill_values[col] = 'missing'
    else:
        fill_values[col] = df[col].median()


In [ ]:
fill_values

In [ ]:
df = df.fillna(fill_values)
df_test = df_test.fillna(fill_values)

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_combined = pd.concat([df, df_test], axis=0)

#Convert all categorical columns to string type to avoid mixed data types
for col in df_combined.select_dtypes(include=['object']).columns:
  df_combined[col] = df_combined[col].astype(str)

#Apply label encoding on combined dataset
label_encoders = {}
for col in df_combined.select_dtypes(include=['object']).columns:
  label_encoders[col] = LabelEncoder()
  df_combined[col] = label_encoders[col].fit_transform(df_combined[col])

  #Store the label encoder for later use
  label_encoders[col] = label_encoders[col]

#Split the combined dataset back into train and test
df_encoded = df_combined[:len(df)]
df_test_encoded = df_combined[len(df):]

In [ ]:
df_test_encoded.drop('SalePrice', axis = 1, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
#Splitting dataset
X = df_encoded.drop('SalePrice', axis = 1)
y = df_encoded['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Hyper parametertuning
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.model_selection import RandomizedSearchCV

models_and_params = {
    "Decision Tree": {
        "model": DecisionTreeRegressor(),
        "params": {
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "Random Forest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 10]
        }
    },
    "Lasso": {
        "model": Lasso(),
        "params": {
            'alpha': [0.01, 0.1, 1, 10, 100]
        }
    },
    "XGBoost": {
        "model": XGBRegressor(random_state=42),
        "params": {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 10],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0]
        }
    }
}

# Loop through models for hyperparameter tuning
best_estimators = {}
results = []

for model_name, model_info in models_and_params.items():
    print(f"Tuning {model_name}...")

    # Initialize RandomizedSearchCV
    search = RandomizedSearchCV(
        estimator=model_info["model"],
        param_distributions=model_info["params"],
        scoring='neg_mean_squared_error',
        cv=5,
        n_iter=10,
        random_state=42,
        n_jobs=-1
    )

    # Fit the model
    search.fit(X_train, y_train)

    # Best estimator and parameters
    best_estimators[model_name] = search.best_estimator_
    best_params = search.best_params_

    # Evaluate on the test set
    y_pred = search.best_estimator_.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store results
    results.append({
        "Model": model_name,
        "Best Parameters": best_params,
        "MSE": mse,
        "R² Score": r2
    })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Display results
print(results_df)

Tuning Decision Tree...
Tuning Random Forest...
Tuning Gradient Boosting...
Tuning Lasso...
Tuning XGBoost...
               Model                                    Best Parameters  \
0      Decision Tree  {'min_samples_split': 2, 'min_samples_leaf': 2...   
1      Random Forest  {'n_estimators': 300, 'min_samples_split': 2, ...   
2  Gradient Boosting  {'n_estimators': 300, 'max_depth': 3, 'learnin...   
3              Lasso                                     {'alpha': 100}   
4            XGBoost  {'subsample': 1.0, 'n_estimators': 300, 'max_d...   

            MSE  R² Score  
0  1.512929e+09  0.802756  
1  8.048747e+08  0.895066  
2  7.702118e+08  0.899586  
3  1.178381e+09  0.846371  
4  7.209749e+08  0.906005  


In [ ]:
result_df

,Model,best_params,mse,r2
0,Decision Tree,"{'min_samples_split': 5, 'min_samples_leaf': 2...",0.0,1.0
1,Random Forest,"{'n_estimators': 200, 'min_samples_split': 2, ...",0.0,1.0
2,Gradient Boosting,"{'n_estimators': 300, 'max_depth': 3, 'learnin...",0.0,1.0
3,Lasso,{'alpha': 10},0.0,1.0
4,XGBoost,"{'subsample': 1.0, 'n_estimators': 200, 'max_d...",0.0,1.0


In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, mean_squared_error

from xgboost import XGBRegressor

xgboost_model = XGBRegressor(
    subsample=1,
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    colsample_bytree=0.9,
    random_state=42
)
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross-validation

# Use cross_val_score to calculate the scores
mse_scores = cross_val_score(
    Xgboost,
    X_train,
    y_train,
    scoring="r2",
    cv=kf
)

# Calculate the mean and standard deviation of the MSE
mean_mse = np.mean(mse_scores)
std_mse = np.std(mse_scores)

print(f"Cross-Validation Mean r2: {mean_mse:.2f}")
print(f"Cross-Validation MSE Std Dev: {std_mse:.2f}")

Cross-Validation Mean r2: 0.84
Cross-Validation MSE Std Dev: 0.04


In [ ]:
X_train = df_encoded.drop('SalePrice', axis=1)
y_train = df_encoded['SalePrice']
X_test = df_test_encoded
X_test = X_test[X_train.columns]

xgboost_model.fit(X_train, y_train)

# Predict on the test dataset
test_predictions = xgboost_model.predict(X_test)

# save predictions
predictions_df = pd.DataFrame({ 'SalesPrice': test_predictions})
predictions_df = predictions_df.iloc[1:]
predictions_df.to_csv('xgboost_predictions.csv', index=False)

print("Training on the full dataset is complete. Predictions saved!")

Training on the full dataset is complete. Predictions saved!
